# ReFuTab ? run the full scaling experiment

**Select Runtime ? Change runtime type ? GPU, then Runtime ? Run all.**

Mount the Drive you want to use. When prompted, enter your GitHub token with **Contents: Read and write** access to `diyarh26/residual-guided-text-tabular` ([GitHub permission documentation](https://docs.github.com/en/rest/repos/contents#create-or-update-file-contents)). No old run folder or file transfer is needed. The notebook installs everything, prepares the data, checks the GPU and starts all 72 fits automatically.

Checkpoints stay in the mounted Drive. Scalar results and CSV scores are pushed automatically to a `results/<run-id>` branch in your GitHub repository during training and at completion. Rerun this same notebook after an interruption to resume. The earlier pilot audit remains a separate pending review.


In [ ]:
import getpass
import json
import os
from pathlib import Path
import re
import subprocess
import sys
from datetime import datetime, timezone
from uuid import uuid4
from google.colab import drive

PROJECT_SHA = "SET_VERIFIED_RUNALL_COMMIT"
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_SHA) is None:
    raise ValueError("Notebook source pin is invalid.")
REPOSITORY = "diyarh26/residual-guided-text-tabular"
PROJECT_DIR = Path("/content/refutab-runall")
SCALING_CONFIG = "configs/phase4_scaling_standalone.json"
drive.mount("/content/drive")
RUNS_ROOT = Path("/content/drive/MyDrive/ReFuTab/standalone_scaling")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
pointer = RUNS_ROOT / "current.json"
if pointer.exists():
    session = json.loads(pointer.read_text())
    if session["source_sha"] != PROJECT_SHA:
        raise RuntimeError("An existing run uses another source version; preserve it before starting a different experiment.")
    RUN_ID = session["run_id"]
else:
    RUN_ID = "scaling-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    with pointer.open("x") as stream:
        json.dump({"run_id": RUN_ID, "source_sha": PROJECT_SHA}, stream)
RUN_DIRECTORY = RUNS_ROOT / RUN_ID
BOOTSTRAP_LOG = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".log")
BOOTSTRAP_LOG.parent.mkdir(parents=True, exist_ok=True)
HARDWARE_RECORD = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".hardware-" + uuid4().hex[:8] + ".json")
CAPACITY_RECORD = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".capacity-" + uuid4().hex[:8] + ".json")
GITHUB_TOKEN = getpass.getpass("GitHub token (Contents: Read and write for your repository): ").strip()
if not GITHUB_TOKEN:
    raise ValueError("A GitHub token is needed to save results to your repository.")
print("Run:", RUN_ID)
print("Checkpoints:", RUN_DIRECTORY)
print("GitHub results:", f"https://github.com/{REPOSITORY}/tree/results/{RUN_ID}/results/scaling_runs/{RUN_ID}")


In [ ]:
import os
import shlex
import subprocess
import sys

def run(command, *, cwd=None):
    # Each command streams to Drive from startup; failures stop the cell.
    # Colab sets an inline backend unavailable in the isolated Python environment.
    child_env = os.environ.copy()
    child_env["MPLBACKEND"] = "Agg"
    # Required by deterministic PyTorch CUDA matrix multiplication.
    child_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    lines = []
    with BOOTSTRAP_LOG.open("a", encoding="utf-8") as stream:
        heading = "$ " + shlex.join([str(part) for part in command])
        print(heading)
        stream.write(heading + "\n")
        stream.write("subprocess_MPLBACKEND=Agg\nsubprocess_CUBLAS_WORKSPACE_CONFIG=:4096:8\n")
        stream.flush()
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=child_env) as process:
            for line in process.stdout:
                print(line, end="")
                stream.write(line)
                stream.flush()
                lines.append(line)
            return_code = process.wait()
        stream.write(f"exit_code={return_code}\n")
        stream.flush()
    if return_code:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {BOOTSTRAP_LOG}")
    return "".join(lines)

if not PROJECT_DIR.exists():
    run(["git", "clone", "https://github.com/" + REPOSITORY + ".git", str(PROJECT_DIR)])
    run(["git", "checkout", "--detach", PROJECT_SHA], cwd=PROJECT_DIR)
if run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR).strip() != PROJECT_SHA:
    raise RuntimeError("Existing checkout has another source version.")
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Source checkout has edits; preserve them before running.")
sys.path.insert(0, str(PROJECT_DIR / "src"))
from refutab.scaling_publish import GitHubResults, export_results
publisher = GitHubResults(GITHUB_TOKEN, REPOSITORY, RUN_ID, PROJECT_SHA)
publisher.publish(export_results(RUN_DIRECTORY, RUN_ID, PROJECT_SHA))
print("GitHub write access verified.")
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.11.24"])
run(["uv", "sync", "--project", "environments/gpu", "--locked", "--python", "3.11"], cwd=PROJECT_DIR)
UV_PYTHON = ["uv", "run", "--project", "environments/gpu", "--frozen", "python"]


In [ ]:
run(["nvidia-smi"], cwd=PROJECT_DIR)
# Upstream TAR checks this variable. Its value selects a real verified device.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU_PREFLIGHT = r"""
import json
import os
import platform
import subprocess
import sys
from pathlib import Path
import psutil
import torch
import torchvision

if sys.platform != "linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 11):
    raise RuntimeError("GPU environment requires Linux x86_64 and Python 3.11.")
if torch.__version__ != "2.7.1+cu126" or torchvision.__version__ != "0.22.1+cu126":
    raise RuntimeError("Unexpected Torch/torchvision build; restore the committed GPU lock.")
if torch.version.cuda != "12.6" or not torch.cuda.is_available():
    raise RuntimeError("A working NVIDIA GPU and compatible driver are required; CPU fallback is disabled.")
if os.environ.get("CUBLAS_WORKSPACE_CONFIG") != ":4096:8":
    raise RuntimeError("Set CUBLAS_WORKSPACE_CONFIG=:4096:8 before starting the GPU Python subprocess.")

torch.cuda.set_device(0)
torch.use_deterministic_algorithms(True)
x = torch.ones((32, 32), device="cuda")
if float((x @ x)[0, 0].item()) != 32.0:
    raise RuntimeError("CUDA matrix multiplication failed.")
linear = torch.nn.Linear(1, 16, device="cuda")
linear(torch.ones((32, 1), device="cuda")).sum().backward()
torch.cuda.synchronize()
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
record = {
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda, "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_name": properties.name, "cuda_compute_capability": [properties.major, properties.minor],
    "cuda_total_memory_bytes": total_bytes, "cuda_free_memory_bytes": free_bytes,
    "host_total_memory_bytes": psutil.virtual_memory().total,
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
    "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "deterministic_linear_backward_passed": True,
    "cuda_matmul_passed": True, "training_executed": False, "service_cost": "unmeasured"
}
with Path(sys.argv[1]).open("x", encoding="utf-8") as stream:
    json.dump(record, stream, indent=2, allow_nan=False)
    stream.write("\n")
print(json.dumps(record, indent=2, allow_nan=False))
"""
run(UV_PYTHON + ["-c", GPU_PREFLIGHT, str(HARDWARE_RECORD)], cwd=PROJECT_DIR)


In [ ]:
run(UV_PYTHON + ["scripts/prepare_phase1.py"], cwd=PROJECT_DIR)
run(UV_PYTHON + ["scripts/prepare_phase4.py"], cwd=PROJECT_DIR)
run(UV_PYTHON + ["-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_DIR)
run(UV_PYTHON + ["scripts/check_tar_memory.py", "--output", str(CAPACITY_RECORD)], cwd=PROJECT_DIR)
run(UV_PYTHON + ["scripts/run_scaling.py", "--config", SCALING_CONFIG], cwd=PROJECT_DIR)


In [ ]:
from refutab.colab_scaling import run_and_publish

command = UV_PYTHON + ["scripts/run_scaling.py", "--config", SCALING_CONFIG, "--launch"]
if (RUN_DIRECTORY / "manifest.json").exists():
    command += ["--resume-run", str(RUN_DIRECTORY)]
else:
    command += ["--run-id", RUN_ID, "--output-root", str(RUNS_ROOT)]
results_url = run_and_publish(
    command, root=PROJECT_DIR, directory=RUN_DIRECTORY, run_id=RUN_ID,
    source_sha=PROJECT_SHA, repository=REPOSITORY, token=GITHUB_TOKEN, console_log=BOOTSTRAP_LOG,
)
print("FINISHED. Results saved:", results_url)
